# **SAM 3 (Segment Anything Model 3) Inference Pipeline for Waste & Conveyor Object Segmentation**

This notebook provides an end-to-end inference and post-processing pipeline using Meta's Segment Anything Model 3 (SAM 3). It supports open-vocabulary grounded segmentation using text prompts (e.g., detecting flat/crumpled packaging, sachets, and pouches on industrial conveyor belts.

# Prerequirements

In [ ]:
# Install Dependencies & Build SAM 3 Repository and RESTART
import sys
!{sys.executable} -m pip install opencv-python matplotlib scikit-learn
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

In [ ]:
# Import Native SAM 3 Components

from sam3 import model_builder as sam3_model_builder
from sam3.model import sam3_image_processor
from sam3.visualization_utils import plot_results

In [ ]:
# Environment Setup & Dependency Verification

import torch
import torchvision
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())
from huggingface_hub import hf_hub_download
import sys
import numpy as np
import torch
from PIL import Image
import sys
import gc
from transformers import Sam3Model, Sam3Processor
from huggingface_hub import snapshot_download
import matplotlib
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Authenticate with Hugging Face Hub

!pip install huggingface-hub
from huggingface_hub import login
login()

In [ ]:
url = (
    "https://raw.githubusercontent.com/tensorflow/models/master/official/"
    "projects/waste_identification_ml/pre_processing/config/sample_images/"
    "IMG_6509.png"
)

!curl -O {url} > /dev/null 2>&1

In [ ]:
#@title utils


# Intermediate state entries dropped after inference to reduce memory
# footprint. They are set by the SAM3 processor but not needed downstream.
_INFERENCE_KEYS_TO_DROP = frozenset(
    ["backbone_out", "geometric_prompt", "image_embeddings"]
)

# State entries that are per-detection arrays; kept in lockstep after any
# filtering step.
_STATE_ARRAY_KEYS = ("masks", "masks_logits", "boxes", "scores")

# State entries preserved unchanged by the edge-visibility filter.
_SAM_META_KEYS = ("original_height", "original_width")


# ── Model setup ──────────────────────────────────────────────────────────────


def build_sam3_processor(checkpoint_path, confidence_threshold):
    """Builds the SAM3 model and its processor.

    Args:
        checkpoint_path: Absolute path to the SAM3 checkpoint.
        confidence_threshold: Minimum confidence passed to the SAM3 processor.

    Returns:
        A SAM3 processor instance ready for inference.
    """
    sam3_model = sam3_model_builder.build_sam3_image_model(
        checkpoint_path=checkpoint_path
    )
    sam3_model.to(device=DEVICE)
    processor = sam3_image_processor.Sam3Processor(
        sam3_model,
        confidence_threshold=confidence_threshold,
    )
    return processor


# ── Preprocess ───────────────────────────────────────────────────────────────


def resize_image_for_inference(image, max_short_side):
    """Resizes an image so its short side does not exceed a maximum length.

    Maintains the original aspect ratio. If the short side is already within
    the limit, the image is returned unchanged.

    Args:
        image: A PIL RGB image to resize.
        max_short_side: Maximum allowed length for the shorter dimension.

    Returns:
        The resized PIL image, or the original if no resize was needed.
    """
    original_width, original_height = image.size
    short_side = min(original_width, original_height)

    if short_side <= max_short_side:
        return image

    scale = max_short_side / short_side
    new_width = int(original_width * scale)
    new_height = int(original_height * scale)

    return image.resize((new_width, new_height), Image.LANCZOS)


# ── Inference ────────────────────────────────────────────────────────────────


def move_inference_state_to_cpu(inference_state):
    """Moves all tensors in an inference state dictionary to CPU.

    Recursively traverses nested dictionaries and moves any torch.Tensor
    values to CPU in place.

    Args:
        inference_state: Dictionary potentially containing tensors and nested
          dictionaries of tensors.

    Returns:
        The same dictionary with all tensors moved to CPU.
    """
    for key, value in inference_state.items():
        if isinstance(value, torch.Tensor):
            inference_state[key] = value.cpu()
        elif isinstance(value, dict):
            move_inference_state_to_cpu(value)
    return inference_state


def run_inference(processor, image, label):
    """Runs SAM3 grounded inference on a single image.

    Performs inference with mixed precision, drops large intermediate tensors
    to free GPU memory, and moves the remaining state to CPU.

    Args:
        processor: SAM3 processor instance.
        image: Input RGB PIL image.
        label: Text prompt for grounded segmentation.

    Returns:
        An inference state dictionary with all tensors on CPU. Keys include
        'masks', 'masks_logits', 'boxes', 'scores', 'original_height',
        'original_width'.
    """
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        state = processor.set_image(image)
        state = processor.set_text_prompt(state=state, prompt=label)

    for key in _INFERENCE_KEYS_TO_DROP:
        state.pop(key, None)

    return move_inference_state_to_cpu(state)


# ── Post-processing filters ──────────────────────────────────────────────────


def filter_contained_sub_masks(state, containment_threshold):
    """Removes smaller masks that are contained within larger masks.

    For each pair of masks, computes the containment ratio
    (intersection / smaller_mask_area). If the ratio exceeds the threshold,
    the smaller mask is discarded. All parallel arrays in state are
    filtered in lockstep.

    Args:
        state: Dict with keys 'masks', 'masks_logits', 'boxes', 'scores'.
          masks is a bool tensor of shape [N, H, W].
        containment_threshold: Ratio above which a smaller mask is considered
          contained and will be removed.

    Returns:
        The filtered state dict with contained masks removed.
    """
    masks = state["masks"]
    num_masks = masks.shape[0]
    if num_masks == 0:
        return state

    flat_masks = masks.view(num_masks, -1).float()
    areas = flat_masks.sum(dim=1)
    pairwise_intersection = flat_masks @ flat_masks.T

    indices_to_remove = set()
    for outer_index in range(num_masks):
        if outer_index in indices_to_remove:
            continue
        for inner_index in range(outer_index + 1, num_masks):
            if inner_index in indices_to_remove:
                continue

            intersection = pairwise_intersection[outer_index, inner_index].item()
            area_outer = areas[outer_index].item()
            area_inner = areas[inner_index].item()

            if area_outer <= area_inner:
                smaller_index = outer_index
                smaller_area = area_outer
            else:
                smaller_index = inner_index
                smaller_area = area_inner

            if smaller_area == 0:
                indices_to_remove.add(smaller_index)
                continue

            containment_ratio = intersection / smaller_area
            if containment_ratio > containment_threshold:
                indices_to_remove.add(smaller_index)

    keep_indices = sorted(set(range(num_masks)) - indices_to_remove)
    keep_tensor = torch.tensor(keep_indices, dtype=torch.long)

    for key in _STATE_ARRAY_KEYS:
        state[key] = state[key][keep_tensor]

    return state


def merge_contained_boxes(state, containment_threshold=0.7):
    """Merges detections where a smaller box is largely contained in a larger.

    Uses containment ratio (intersection_area / smaller_box_area) instead
    of IoU to avoid merging adjacent objects whose boxes partially overlap.

    Args:
        state: SAM output dict with 'masks', 'boxes', 'scores' keys.
        containment_threshold: Minimum fraction of the smaller box's area that
          must overlap with the larger box to trigger a merge.

    Returns:
        A state dict with merged detections.
    """
    masks = state["masks"]
    boxes = state["boxes"]
    scores = state["scores"]

    if len(scores) == 0:
        return state

    num_detections = len(masks)
    box_areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

    is_absorbed = torch.zeros(num_detections, dtype=torch.bool)
    absorb_target = list(range(num_detections))

    for outer_index in range(num_detections):
        if is_absorbed[outer_index]:
            continue
        for inner_index in range(outer_index + 1, num_detections):
            if is_absorbed[inner_index]:
                continue

            intersection_x_min = torch.max(
                boxes[outer_index, 0], boxes[inner_index, 0]
            )
            intersection_y_min = torch.max(
                boxes[outer_index, 1], boxes[inner_index, 1]
            )
            intersection_x_max = torch.min(
                boxes[outer_index, 2], boxes[inner_index, 2]
            )
            intersection_y_max = torch.min(
                boxes[outer_index, 3], boxes[inner_index, 3]
            )

            intersection_area = torch.clamp(
                intersection_x_max - intersection_x_min, min=0
            ) * torch.clamp(intersection_y_max - intersection_y_min, min=0)

            if box_areas[outer_index] <= box_areas[inner_index]:
                smaller_index = outer_index
                larger_index = inner_index
                smaller_area = box_areas[outer_index]
            else:
                smaller_index = inner_index
                larger_index = outer_index
                smaller_area = box_areas[inner_index]

            if smaller_area == 0:
                is_absorbed[smaller_index] = True
                continue

            containment_ratio = intersection_area / smaller_area
            if containment_ratio > containment_threshold:
                is_absorbed[smaller_index] = True
                absorb_target[smaller_index] = larger_index

    # Group absorbed detections with their targets.
    groups = {}
    for detection_index in range(num_detections):
        if is_absorbed[detection_index]:
            target = absorb_target[detection_index]
            if target not in groups:
                groups[target] = [target]
            groups[target].append(detection_index)
        elif detection_index not in groups:
            groups[detection_index] = [detection_index]

    merged_masks = []
    merged_boxes = []
    merged_scores = []

    for member_indices in groups.values():
        member_tensor = torch.tensor(member_indices, dtype=torch.long)

        union_mask = masks[member_tensor].squeeze(1).any(dim=0)

        group_boxes = boxes[member_tensor]
        enclosing_box = torch.stack([
            group_boxes[:, 0].min(),
            group_boxes[:, 1].min(),
            group_boxes[:, 2].max(),
            group_boxes[:, 3].max(),
        ])

        combined_score = torch.tensor(
            min(scores[member_tensor].sum().item(), 1.0)
        )

        merged_masks.append(union_mask)
        merged_boxes.append(enclosing_box)
        merged_scores.append(combined_score)

    state["masks"] = torch.stack(merged_masks).unsqueeze(1)
    state["boxes"] = torch.stack(merged_boxes)
    state["scores"] = torch.stack(merged_scores)

    return state


def get_valid_bottle_indices(
    sam_output, margin=5, visibility_threshold=0.5
):
    """Filters SAM output to remove edge detections less than 50% visible.

    Detections fully inside the image are always kept. Detections touching
    the image edge are kept only if their mask area is at least
    visibility_threshold * median_area of the inner detections.

    Args:
        sam_output: SAM output dict with keys 'boxes', 'masks',
          'masks_logits', 'scores', 'original_height', 'original_width'.
        margin: Pixel margin from the image border to consider as edge.
        visibility_threshold: Minimum fraction of the median inner-detection
          area required for an edge detection to be kept.

    Returns:
        A filtered SAM output dict with partially visible edge detections
        removed.
    """
    boxes = sam_output["boxes"].numpy()
    masks = sam_output["masks"].numpy()
    if masks.ndim == 4:
        masks = masks.squeeze(1)

    image_height = sam_output["original_height"]
    image_width = sam_output["original_width"]

    inner_indices = []
    edge_indices = []
    for detection_index, (x_min, y_min, x_max, y_max) in enumerate(boxes):
        touches_edge = (
            x_min <= margin
            or y_min <= margin
            or x_max >= image_width - margin
            or y_max >= image_height - margin
        )
        if touches_edge:
            edge_indices.append(detection_index)
        else:
            inner_indices.append(detection_index)

    if not inner_indices:
        return sam_output

    inner_areas = [np.sum(masks[i]) for i in inner_indices]
    median_area = np.median(inner_areas)
    minimum_valid_area = visibility_threshold * median_area

    valid_edge_indices = [
        i for i in edge_indices if np.sum(masks[i]) >= minimum_valid_area
    ]

    valid_indices = sorted(inner_indices + valid_edge_indices)

    filtered_output = {}
    for key in _SAM_META_KEYS:
        filtered_output[key] = sam_output[key]
    for key in _STATE_ARRAY_KEYS:
        filtered_output[key] = sam_output[key][valid_indices]

    return filtered_output


# ── End-to-end wrapper ───────────────────────────────────────────────────────


def sam3_detect(
    image,
    processor,
    prompt,
    max_short_side=1024,
    containment_threshold=0.98,
    score_threshold=0.0,
    merge_boxes_for_packets=True,
):
    """Runs SAM3 inference plus the full post-processing pipeline.

    Applies, in order: image resize, SAM3 inference, contained sub-mask
    removal, contained-box merge (only when prompt == 'packets' and
    merge_boxes_for_packets is True), edge-visibility filter, and a final
    score-threshold filter.

    Args:
        image: Input RGB PIL image.
        processor: SAM3 processor instance from build_sam3_processor.
        prompt: Text prompt for grounded segmentation.
        max_short_side: Maximum allowed length for the shorter image
          dimension at inference time.
        containment_threshold: Ratio above which a smaller mask is treated
          as contained by a larger one and removed.
        score_threshold: Minimum score for a detection to be kept in the
          final output. Set to 0.0 to keep all detections.
        merge_boxes_for_packets: If True and prompt == 'packets', runs the
          contained-box merge step.

    Returns:
        A state dict with keys 'masks', 'masks_logits', 'boxes', 'scores',
        'original_height', 'original_width', containing only detections
        that passed every filter.
    """
    image = resize_image_for_inference(image, max_short_side=max_short_side)

    state = run_inference(processor, image, prompt)
    if not state["scores"].tolist():
        return state

    state = filter_contained_sub_masks(
        state, containment_threshold=containment_threshold
    )
    if prompt == "packets" and merge_boxes_for_packets:
        state = merge_contained_boxes(state)
    state = get_valid_bottle_indices(state)

    if score_threshold > 0.0:
        keep_mask = state["scores"] >= score_threshold
        for key in _STATE_ARRAY_KEYS:
            state[key] = state[key][keep_mask]

    return state


def cleanup_memory():
    """
    Clean up GPU and CPU memory.
    Call this after each inference or batch of inferences.

    Usage:
        # After your inference code
        cleanup_memory()
    """
    # Force garbage collection
    gc.collect()

    # Clear CUDA cache if GPU is available
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()  # Wait for all operations to finish

    print("Memory cleaned up successfully")


def print_memory_stats():
    """
    Print current memory usage (CPU and GPU).

    Usage:
        print_memory_stats()
    """
    import psutil
    import os

    # CPU Memory
    process = psutil.Process(os.getpid())
    cpu_mem_gb = process.memory_info().rss / 1024**3

    print(f"CPU RAM: {cpu_mem_gb:.2f} GB")

    # GPU Memory
    if torch.cuda.is_available():
        gpu_mem_gb = torch.cuda.memory_allocated() / 1024**3
        gpu_cached_gb = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Allocated: {gpu_mem_gb:.2f} GB")
        print(f"GPU Cached: {gpu_cached_gb:.2f} GB")
    else:
        print("GPU: Not available")


def overlay_masks(image, masks):
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)

    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [
        tuple(int(c * 255) for c in cmap(i)[:3])
        for i in range(n_masks)
    ]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image

# Inference with original weights

In [ ]:
# Download raw weights from facebook

files_to_download = [
    "sam3.pt",
    "config.json",
    # Add any other specific files you know you need
]

# Download specific files only.
for filename in files_to_download:
    hf_hub_download(
        repo_id="facebook/sam3",
        filename=filename,
        local_dir="./sam3_weights/",
        local_dir_use_symlinks=False
    )

In [ ]:
# Load image and resize for faster inference

image = Image.open("/content/IMG_6509.png").convert("RGB")
image_resized = resize_image_for_inference(image, max_short_side=1024)

In [ ]:
# Load the SAM3 model

processor = build_sam3_processor(checkpoint_path="/content/sam3_weights/sam3.pt",confidence_threshold=0.3)

In [ ]:
# Run inference and plot results

state = sam3_detect(
    image,
    processor,
    prompt="pouch or packet or wrapper or crumpled paper or sachet or paper",
    score_threshold=0.0,
)

plot_results(image_resized, state)

In [ ]:
print_memory_stats()  # Check initial memory
cleanup_memory()
print('\nMemory after clean up')
print_memory_stats()  # Check memory after clean up

# Inference with Hugging Face Transformer weights

In [ ]:
# This pattern downloads only the files needed for inference
# and skips the duplicate "sam3.pt" file (saving ~3.5 GB)
allowed_files = [
    "*.json",        # config.json, tokenizer.json, preprocessor_config.json
    "*.safetensors", # The model weights (optimized for HF)
    "*.txt"          # merges.txt (needed for the text tokenizer)
]


# Download ALL necessary files (not just sam3.pt)
model_path = snapshot_download(
    repo_id="facebook/sam3",
    local_dir="./huggingface_sam3_weights/",
    allow_patterns=allowed_files,
    local_dir_use_symlinks=False
)

print("Model saved at:", model_path)

In [ ]:
label_to_predict = "pouch or packet or wrapper or crumpled paper or sachet or paper"

image = Image.open("/content/img_20260805_171747_926.jpg").convert("RGB")
image_resized = resize_image_for_inference(image, max_short_side=1024)
image_resized.mode

In [ ]:
# Load model

hf_model = Sam3Model.from_pretrained(
    "/content/huggingface_sam3_weights",
    local_files_only=True,
    torch_dtype=torch.float16
).to(DEVICE)

hf_processor = Sam3Processor.from_pretrained(
    "/content/huggingface_sam3_weights",
    local_files_only=True
)

In [ ]:
# Tokenization, Forward Pass & Overlay Visualization

inputs = hf_processor(images=image, text=label_to_predict, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = hf_model(**inputs)

print("Prediction successful!")
print("Output keys:", outputs.keys())

# Post-process results
results = hf_processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

overlay_masks(image, results['masks'])

In [ ]:
print_memory_stats()  # Check initial memory
cleanup_memory()
print('\nMemory after clean up')
print_memory_stats()  # Check memory after clean up